In [12]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
SCRIPT_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, "data")
PROCESSED_DATA_DIR_PATH = os.path.join(DATA_DIR_PATH, "processed_data")

In [4]:
# import IEA scored data
policy_df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR_PATH, "IEA_scored_cpsi.csv"))
policy_df.head()

,iso3,year,avg_score,policy_count,dominant_topic,dominant_category,CPSI
0,AFG,1971,0.75,1,Energy,Solar Power,0.51986
1,AFG,1972,0.75,1,Energy,Solar Power,0.51986
2,AFG,1973,0.75,1,Energy,Solar Power,0.51986
3,AFG,1974,0.75,1,Energy,Solar Power,0.51986
4,AFG,1975,0.75,1,Energy,Solar Power,0.51986


In [6]:
policy_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9255 entries, 0 to 9254
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   iso3               9255 non-null   object 
 1   year               9255 non-null   int64  
 2   avg_score          9255 non-null   float64
 3   policy_count       9255 non-null   int64  
 4   dominant_topic     9255 non-null   object 
 5   dominant_category  9255 non-null   object 
 6   CPSI               9255 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 506.3+ KB


In [5]:
# import emissions data
emissions_df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR_PATH, "total_emissions_db.csv"))
emissions_df.head()

,Code,Income group,Year,Total Emissions
0,ABW,High income,2000,0.335765
1,ABW,High income,2001,0.344135
2,ABW,High income,2002,0.363222
3,ABW,High income,2003,0.412246
4,ABW,High income,2004,0.430187


In [8]:
# Rename cols in emissions_df
emissions_df.columns = emissions_df.columns.str.lower().str.replace(' ', '_')
emissions_df.rename(columns={'code': 'iso3'}, inplace=True)
emissions_df.head()

,iso3,income_group,year,total_emissions
0,ABW,High income,2000,0.335765
1,ABW,High income,2001,0.344135
2,ABW,High income,2002,0.363222
3,ABW,High income,2003,0.412246
4,ABW,High income,2004,0.430187


In [9]:
emissions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4485 entries, 0 to 4484
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   iso3             4485 non-null   object 
 1   income_group     4485 non-null   object 
 2   year             4485 non-null   int64  
 3   total_emissions  4485 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 140.3+ KB


In [10]:
policy_emissions_df = pd.merge(policy_df, emissions_df, on=["iso3", "year"], how="inner")
policy_emissions_df.head()

,iso3,year,avg_score,policy_count,dominant_topic,dominant_category,CPSI,income_group,total_emissions
0,AFG,2000,0.75,1,Energy,Solar Power,0.51986,Low income,25.390391
1,AFG,2001,0.75,1,Energy,Solar Power,0.51986,Low income,23.723115
2,AFG,2002,0.75,1,Energy,Solar Power,0.51986,Low income,26.383509
3,AFG,2003,0.75,1,Energy,Solar Power,0.51986,Low income,27.071538
4,AFG,2004,0.75,1,Energy,Solar Power,0.51986,Low income,27.128799


In [11]:
policy_emissions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4187 entries, 0 to 4186
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   iso3               4187 non-null   object 
 1   year               4187 non-null   int64  
 2   avg_score          4187 non-null   float64
 3   policy_count       4187 non-null   int64  
 4   dominant_topic     4187 non-null   object 
 5   dominant_category  4187 non-null   object 
 6   CPSI               4187 non-null   float64
 7   income_group       4187 non-null   object 
 8   total_emissions    4187 non-null   float64
dtypes: float64(3), int64(2), object(4)
memory usage: 294.5+ KB


In [25]:
oecd_countries_iso3 = [
    "AUS",  # Australia
    "AUT",  # Austria
    "BEL",  # Belgium
    "CAN",  # Canada
    "CHL",  # Chile
    "COL",  # Colombia
    "CZE",  # Czech Republic
    "DNK",  # Denmark
    "EST",  # Estonia
    "FIN",  # Finland
    "FRA",  # France
    "DEU",  # Germany
    "GRC",  # Greece
    "HUN",  # Hungary
    "ISL",  # Iceland
    "IRL",  # Ireland
    "ISR",  # Israel
    "ITA",  # Italy
    "JPN",  # Japan
    "KOR",  # South Korea
    "LVA",  # Latvia
    "LTU",  # Lithuania
    "LUX",  # Luxembourg
    "MEX",  # Mexico
    "NLD",  # Netherlands
    "NZL",  # New Zealand
    "NOR",  # Norway
    "POL",  # Poland
    "PRT",  # Portugal
    "SVK",  # Slovakia
    "SVN",  # Slovenia
    "ESP",  # Spain
    "SWE",  # Sweden
    "CHE",  # Switzerland
    "TUR",  # Turkey
    "GBR",  # United Kingdom
    "USA"   # United States
]

## EDA

In [ ]:
# Function to calculate the correlation, plot the relation, and add a regression line
def plot_correlation(df, col1, col2, iso3=None):
    if iso3:
        df = df[df['iso3'] == iso3]
        
    # Calculate the correlation
    correlation = df[col1].corr(df[col2])
        
    # Plotting the relation
    sns.regplot(x=col1, y=col2, data=df, ci=None, line_kws={"color": "red"})
    plt.title(f"Scatter plot of {col1} vs {col2} for {iso3 if iso3 else 'all countries'}\nCorrelation: {correlation:.2f}")
    plt.grid()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.xlabel(col1)
    plt.ylabel(col2)
    plt.show()


In [31]:
# Function to calculate the correlation, plot the relation, and add regression lines for multiple countries
def plot_correlation_multiple(df, col1, col2, iso3_list):
    n = len(iso3_list)
    ncols = 3  # Number of columns in the subplot grid
    nrows = (n + ncols - 1) // ncols  # Calculate required number of rows

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 4 * nrows))
    axes = axes.flatten()  # Flatten in case of a multi-row, multi-column grid

    for i, iso3 in enumerate(iso3_list):
        ax = axes[i]
        df_filtered = df[df['iso3'] == iso3]

        if df_filtered.empty:
            ax.set_title(f"No data for {iso3}")
            ax.axis('off')
            continue

        correlation = df_filtered[col1].corr(df_filtered[col2])
        sns.regplot(x=col1, y=col2, data=df_filtered, ci=None, line_kws={"color": "red"}, ax=ax)
        ax.set_title(f"{iso3}\nCorrelation: {correlation:.2f}")
        ax.set_xlabel(col1)
        ax.set_ylabel(col2)
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True)

    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

In [35]:
def calculate_corr_coef_for_each_country(df, col1, col2, iso3_list):
    correlation_results = []
    for iso3 in iso3_list:
        df_filtered = df[df['iso3'] == iso3]
        if not df_filtered.empty:
            correlation = df_filtered[col1].corr(df_filtered[col2])
        else:
            correlation = None
        correlation_results.append({'iso3': iso3, 'correlation': correlation})
    
    return pd.DataFrame(correlation_results)

In [ ]:
plot_correlation_multiple(policy_emissions_df, "CPSI", "total_emissions", oecd_countries_iso3)

In [41]:
countries_corr_df = calculate_corr_coef_for_each_country(policy_emissions_df, "CPSI", "total_emissions", policy_emissions_df['iso3'].unique())
countries_corr_df = countries_corr_df.dropna()

/home/tony-ubuntu/anaconda3/envs/cpia_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/tony-ubuntu/anaconda3/envs/cpia_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [48]:
strong_corr_countries_df = countries_corr_df[(countries_corr_df['correlation'] < -0.5) & (countries_corr_df['correlation'] > -1.0)]
strong_corr_countries_df = strong_corr_countries_df.sort_values(by='correlation', ascending=True)
strong_corr_countries_df = strong_corr_countries_df.reset_index(drop=True)
strong_corr_countries_df

,iso3,correlation
0,DNK,-0.963180
1,GBR,-0.952600
2,DEU,-0.949954
3,FRA,-0.920495
4,GRC,-0.915197
5,USA,-0.912663
6,IRL,-0.895318
7,BEL,-0.893942
8,ITA,-0.877673
9,FIN,-0.856656


## Data preprocessing